In [ ]:
!pip install -q datasets

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

Using Device: cpu


In [ ]:
print("Loading AG News Dataset...")

dataset = load_dataset("fancyzhx/ag_news")

train_data = dataset["train"]
test_data = dataset["test"]

print("Training Samples :", len(train_data))
print("Testing Samples  :", len(test_data))

Loading AG News Dataset...


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Training Samples : 120000
Testing Samples  : 7600


In [ ]:
print("Building Vocabulary...")

vocab = {
    "<unk>": 0,
    "<pad>": 1
}

def tokenize(text):
    return text.lower().split()

for article in train_data["text"]:
    words = tokenize(article)

    for word in words:
        if word not in vocab:
            vocab[word] = len(vocab)

print("Vocabulary Size:", len(vocab))

PAD_IDX = vocab["<pad>"]

Building Vocabulary...
Vocabulary Size: 158735


In [ ]:
def numericalize(text):
    words = tokenize(text)

    return [
        vocab.get(word, vocab["<unk>"])
        for word in words
    ]

In [ ]:
def collate_batch(batch):

    text_list = []
    label_list = []
    length_list = []

    for item in batch:

        text_tensor = torch.tensor(
            numericalize(item["text"]),
            dtype=torch.long
        )

        text_list.append(text_tensor)
        label_list.append(item["label"])
        length_list.append(len(text_tensor))

    padded_texts = pad_sequence(
        text_list,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return (
        padded_texts.to(device),
        torch.tensor(label_list).to(device),
        torch.tensor(length_list)
    )

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch
)

test_loader = DataLoader(
    test_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch
)

In [ ]:
class SimpleRNNClassifier(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            100,
            padding_idx=PAD_IDX
        )

        self.rnn = nn.RNN(
            input_size=100,
            hidden_size=128,
            batch_first=True
        )

        self.fc = nn.Linear(
            128,
            4
        )

    def forward(self, text, lengths):

        embedded = self.embedding(text)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, hidden = self.rnn(packed)

        hidden = hidden.squeeze(0)

        output = self.fc(hidden)

        return output

In [ ]:
model = SimpleRNNClassifier(
    vocab_size=len(vocab)
).to(device)

print(model)

SimpleRNNClassifier(
  (embedding): Embedding(158735, 100, padding_idx=1)
  (rnn): RNN(100, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=4, bias=True)
)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


In [ ]:
def train():

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for texts, labels, lengths in train_loader:

        optimizer.zero_grad()

        outputs = model(texts, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    accuracy = 100 * correct / total

    return total_loss, accuracy

In [ ]:
def evaluate():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for texts, labels, lengths in test_loader:

            outputs = model(texts, lengths)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

    accuracy = 100 * correct / total

    return accuracy

In [ ]:
EPOCHS = 5

print("\nStarting Training...\n")

for epoch in range(EPOCHS):

    train_loss, train_acc = train()

    test_acc = evaluate()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {train_loss:.4f} "
        f"Train Accuracy: {train_acc:.2f}% "
        f"Test Accuracy: {test_acc:.2f}%"
    )



Starting Training...

Epoch [1/5] Loss: 1441.3797 Train Accuracy: 68.37% Test Accuracy: 60.21%
Epoch [2/5] Loss: 1466.5920 Train Accuracy: 68.05% Test Accuracy: 75.16%
Epoch [3/5] Loss: 943.8214 Train Accuracy: 80.98% Test Accuracy: 82.58%
Epoch [4/5] Loss: 821.6943 Train Accuracy: 84.54% Test Accuracy: 75.20%
Epoch [5/5] Loss: 804.4944 Train Accuracy: 84.78% Test Accuracy: 83.67%


In [19]:
import torch

torch.save(
    model.state_dict(),
    "ag_news_rnn.pth"
)

print("\nModel Saved Successfully!")


Model Saved Successfully!


In [20]:
categories = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

def predict_news(text):

    model.eval()

    tokens = torch.tensor(
        [numericalize(text)],
        dtype=torch.long
    ).to(device)

    lengths = torch.tensor(
        [tokens.size(1)]
    )

    with torch.no_grad():

        output = model(tokens, lengths)

        prediction = output.argmax(dim=1).item()

    return categories[prediction]

In [21]:
sample_news = """
Apple introduced a new AI-powered smartphone
with advanced machine learning features.
"""

prediction = predict_news(sample_news)

print("\nSample News:")
print(sample_news)

print("\nPredicted Category:", prediction)


Sample News:

Apple introduced a new AI-powered smartphone
with advanced machine learning features.


Predicted Category: Sci/Tech
